# Augment dents from ml_data_v4 (same logic as augment.ipynb)

**Paired std + patch:** For each sample whose stem exists in **both** `dent/{iid}/std_normalized` and `patch_normalized`, the same `np.roll` augments are applied to **std** and **patch** separately (patch outputs are always real patch arrays).

**Run order:** (1) imports/paths, (2) create folders, (3) roll augmentation from `dent/`, (4) **noise + scale** on `pos/` (same as former `dataset_v4` cell). Re-run (3)/(4) after changing parameters.

Output: **only** under `data/ml_data_v4/pos/` (`std_normalized/`, `patch_normalized/`, each with `qc/`). **`dent/` is read-only** (sources); this notebook must not write anything under `ml_data_v4/dent/`.

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

# Paths (Linux; WSL UNC \\wsl...\home\... -> /home/...)
BASE = Path("/home/zmirikha/Github/rnd_q2/data")
ML_DATA_V4 = BASE / "ml_data_v4"
# Read-only source (never write here)
DENT_ROOT = ML_DATA_V4 / "dent"
# All new .npy / .png from this notebook go here only (not under dent/)
AUG_OUT = ML_DATA_V4 / "pos"

INSPECTION_IDS = ["0A1OLRV96N3", "0A49KLT3B7Y", "0AGBXB4WEGN", "0ABP0TFUSH1"]

# Same roll parameters as augment.ipynb
VERTICAL_SHIFTS = [2, 4, 6, 8, 10, 12, 14, 16, 18]
HORIZONTAL_SHIFTS = [0]#[50, 85, -50, -85]


def save_qc_png(arr: np.ndarray, path: Path) -> None:
    """Save array as a QC image (transposed, inferno, no axes).

    Shared by the roll-augmentation cell and the noise/scale cell so both
    produce visually identical QC PNGs.
    """
    plt.imshow(
        arr.T, cmap="inferno", origin="lower", aspect="auto",
        interpolation="nearest",
    )
    plt.axis("off")
    plt.savefig(path, bbox_inches="tight", pad_inches=0)
    plt.close()


def _save_if_missing(arr: np.ndarray, npy_path: Path, png_path: Path) -> None:
    """Write NPY + QC PNG only if missing, so re-running is cheap and safe."""
    if not npy_path.exists():
        np.save(npy_path, arr)
    if not png_path.exists():
        save_qc_png(arr, png_path)

In [2]:
# Create pos/ layout: std_normalized, patch_normalized, each with qc subfolder
for norm in ["std_normalized", "patch_normalized"]:
    (AUG_OUT / norm).mkdir(parents=True, exist_ok=True)
    (AUG_OUT / norm / "qc").mkdir(parents=True, exist_ok=True)
print("Created:", AUG_OUT)
for norm in ["std_normalized", "patch_normalized"]:
    print(f"  {norm}/, {norm}/qc/")

Created: /home/zmirikha/Github/rnd_q2/data/ml_data_v4/pos
  std_normalized/, std_normalized/qc/
  patch_normalized/, patch_normalized/qc/


In [3]:
def parse_stem_labels(stem: str) -> tuple[str, list[str]]:
    """Split stem into base_id and list of label strings. E.g. d0014950856_['015','016'] -> ('d0014950856', ['015','016'])."""
    import ast
    if "_" not in stem:
        return stem, []
    base_id, _, labels_part = stem.partition("_")
    try:
        labels = ast.literal_eval(labels_part.replace("'", '"'))
        if isinstance(labels, list):
            labels = [str(x) for x in labels]
        else:
            labels = []
    except Exception:
        labels = []
    return base_id, labels


def shift_labels(labels: list[str], shift_v: int, mod: int = 22) -> list[str]:
    """Apply vertical roll shift to label indices (tracks 0..21); wrap with mod 22."""
    if not labels:
        return labels
    out = []
    for s in labels:
        try:
            n = (int(s) + shift_v) % mod
            out.append(f"{n:03d}")
        except ValueError:
            out.append(s)
    return out


def labels_to_str(labels: list[str]) -> str:
    """Format labels for filename: ['015','016'] -> \"['015','016']\" (no spaces, single quotes)."""
    return str(labels).replace(" ", "").replace('"', "'")


def _roll2d(arr: np.ndarray, shift_v: int, shift_h: int) -> np.ndarray:
    out = np.roll(arr, shift_v, axis=0)
    return np.roll(out, shift_h, axis=1)


def augment_dents_paired() -> tuple[int, int, int, int]:
    """For each dent sample that exists in BOTH std_normalized and patch_normalized, apply the same rolls to each modality.

    Reads from DENT_ROOT/<iid>/ only. Writes only under AUG_OUT (pos/), never under dent/.
    Returns (n_paired_originals, n_paired_augmented, n_std_only, n_patch_only) for diagnostics.
    """
    out_std = AUG_OUT / "std_normalized"
    out_std_qc = out_std / "qc"
    out_patch = AUG_OUT / "patch_normalized"
    out_patch_qc = out_patch / "qc"
    _aug = AUG_OUT.resolve()
    assert out_std.resolve().is_relative_to(_aug) and out_patch.resolve().is_relative_to(_aug)
    for d in (out_std, out_std_qc, out_patch, out_patch_qc):
        d.mkdir(parents=True, exist_ok=True)

    n_orig = n_aug = 0
    n_std_only = n_patch_only = 0
    for iid in INSPECTION_IDS:
        std_dir = DENT_ROOT / iid / "std_normalized"
        patch_dir = DENT_ROOT / iid / "patch_normalized"
        if not std_dir.exists() or not patch_dir.exists():
            print(f"  skip {iid}: missing std_dir or patch_dir")
            continue
        std_stems = {p.stem for p in std_dir.glob("*.npy")}
        patch_stems = {p.stem for p in patch_dir.glob("*.npy")}
        common = sorted(std_stems & patch_stems)
        only_std = std_stems - patch_stems
        only_patch = patch_stems - std_stems
        if only_std:
            n_std_only += len(only_std)
            print(f"  {iid}: {len(only_std)} .npy only in std_normalized (not augmented to patch folder)")
        if only_patch:
            n_patch_only += len(only_patch)
            print(f"  {iid}: {len(only_patch)} .npy only in patch_normalized (not augmented to std folder)")

        for stem in common:
            base_id, labels = parse_stem_labels(stem)
            orig_std_npy = out_std / f"{stem}.npy"
            orig_std_png = out_std_qc / f"{stem}.png"
            orig_patch_npy = out_patch / f"{stem}.npy"
            orig_patch_png = out_patch_qc / f"{stem}.png"

            roll_targets = []
            for shift_v in VERTICAL_SHIFTS:
                new_labels = shift_labels(labels, shift_v)
                new_labels_str = labels_to_str(new_labels) if new_labels else (labels_to_str(labels) if labels else "")
                aug_base = f"{base_id}_{new_labels_str}" if new_labels_str else base_id
                for shift_h in HORIZONTAL_SHIFTS:
                    aug_stem = f"{aug_base}_vroll_{shift_v}_hroll_{shift_h}"
                    roll_targets.append((
                        shift_v, shift_h,
                        out_std / f"{aug_stem}.npy",
                        out_std_qc / f"{aug_stem}.png",
                        out_patch / f"{aug_stem}.npy",
                        out_patch_qc / f"{aug_stem}.png",
                    ))

            originals_done = all(
                p.exists() for p in (orig_std_npy, orig_std_png, orig_patch_npy, orig_patch_png)
            )
            rolls_done = all(
                std_n.exists() and std_p.exists() and patch_n.exists() and patch_p.exists()
                for _, _, std_n, std_p, patch_n, patch_p in roll_targets
            )
            n_orig += 1
            n_aug += len(roll_targets)
            if originals_done and rolls_done:
                continue

            pos_std = np.load(std_dir / f"{stem}.npy")
            pos_patch = np.load(patch_dir / f"{stem}.npy")
            _save_if_missing(pos_std, orig_std_npy, orig_std_png)
            _save_if_missing(pos_patch, orig_patch_npy, orig_patch_png)
            for shift_v, shift_h, std_n, std_p, patch_n, patch_p in roll_targets:
                if std_n.exists() and std_p.exists() and patch_n.exists() and patch_p.exists():
                    continue
                r_std = _roll2d(pos_std, shift_v, shift_h)
                r_patch = _roll2d(pos_patch, shift_v, shift_h)
                _save_if_missing(r_std, std_n, std_p)
                _save_if_missing(r_patch, patch_n, patch_p)
    return n_orig, n_aug, n_std_only, n_patch_only


# Run paired augmentation (same cell as definitions — avoids NameError if cells run out of order)
print("Processing paired std + patch augmentation...")
_n_orig, _n_aug, _n_std_only, _n_patch_only = augment_dents_paired()
print(f"  Paired samples: {_n_orig} originals, {_n_aug} roll-augmented pairs (std + patch each)")
print(f"  Unpaired: {_n_std_only} std-only stems, {_n_patch_only} patch-only stems (see messages above)")
print("Done. Check ml_data_v4/pos/std_normalized/ and ml_data_v4/pos/patch_normalized/.")

Processing paired std + patch augmentation...
  Paired samples: 1760 originals, 15840 roll-augmented pairs (std + patch each)
  Unpaired: 0 std-only stems, 0 patch-only stems (see messages above)
Done. Check ml_data_v4/pos/std_normalized/ and ml_data_v4/pos/patch_normalized/.


In [4]:
# Summary
print("pos/ (augmented dents) layout:")
for norm in ["std_normalized", "patch_normalized"]:
    npy_dir = AUG_OUT / norm
    qc_dir = AUG_OUT / norm / "qc"
    n_npy = len(list(npy_dir.glob("*.npy"))) if npy_dir.exists() else 0
    n_png = len(list(qc_dir.glob("*.png"))) if qc_dir.exists() else 0
    print(f"  {norm}: {n_npy} NPYs, qc/: {n_png} PNGs")

pos/ (augmented dents) layout:
  std_normalized: 17600 NPYs, qc/: 17600 PNGs
  patch_normalized: 17600 NPYs, qc/: 17600 PNGs


## Gaussian noise & intensity scaling (moved from `dataset_v4.ipynb`)

Run **after** roll augmentation. Scans **`pos/std_normalized`** and **`pos/patch_normalized`** (flat `*.npy`, not under `dent/`).

Per source file (skipping stems already containing `_aug_noise` or `_aug_scale_`):

- **`_aug_noise`**: Gaussian noise (`NOISE_SIGMA`).
- **`_aug_scale_<factor>`**: intensity in `[SCALE_LOW, SCALE_HIGH]`, clip to original min/max.

**Modality coupling:** the RNG is keyed per stem (via `_stem_rngs`, which spawns two independent streams — one for noise, one for scale), so `std_normalized/<stem>` and `patch_normalized/<stem>` get the **same** scale factor (and same-distribution noise). Their `_aug_scale_<f>` filenames stay paired 1:1 across modalities, regardless of file iteration order or array shape differences.

Re-run safe: existing noise/scale outputs are skipped.

**Tip:** Re-run the **Summary** cell (NPY counts) after this if you want updated totals.

In [5]:
import hashlib
import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

# Requires AUG_OUT from the first path cell (ml_data_v4/pos)
# Match dataset_v4 / create_dataset_v1.1-style augmentations on pos/ only
NOISE_SIGMA = 0.02  # typical 0.01–0.05
SCALE_LOW, SCALE_HIGH = 0.9, 1.1
AUGMENT_SEED = 42


def _skip_noise_scale_stem(stem: str) -> bool:
    return "_aug_noise" in stem or "_aug_scale_" in stem


def _stem_rngs(seed: int, stem: str) -> tuple[np.random.Generator, np.random.Generator]:
    """Per-stem (noise_rng, scale_rng) pair — two independent streams.

    Keying by (seed, stem) instead of consuming one global generator in
    iteration order makes std_normalized/<stem> and patch_normalized/<stem>
    receive identical augmentation parameters for the same stem (so their
    `_aug_scale_<f>` filenames pair 1:1 across modalities). Splitting into two
    spawned streams keeps the SCALE draw shape-independent — std and patch
    arrays often have different shapes, which would otherwise advance the
    shared RNG by different amounts during the noise step. Uses blake2b for a
    stable, process-independent seed.
    """
    digest = hashlib.blake2b(
        stem.encode("utf-8"), digest_size=8, person=b"augv4nse"
    ).digest()
    base = int.from_bytes(digest, "big") ^ (int(seed) & 0xFFFFFFFFFFFFFFFF)
    ss_noise, ss_scale = np.random.SeedSequence(base).spawn(2)
    return np.random.default_rng(ss_noise), np.random.default_rng(ss_scale)


def augment_pos_noise_and_scale(aug_out: Path, seed: int = 42) -> tuple[int, int]:
    """Write only under aug_out (pos/).

    For each non-aug NPY stem, generates one Gaussian-noise file and one
    intensity-scaled file using a per-stem RNG pair (see `_stem_rngs`). This
    makes std_normalized and patch_normalized receive the SAME scale factor
    (and same-distribution noise) for the same stem, keeping the two
    modalities 1:1 by filename after this stage.
    """
    n_noise, n_scale = 0, 0
    _root = aug_out.resolve()
    for norm_name in ["std_normalized", "patch_normalized"]:
        npy_dir = aug_out / norm_name
        qc_dir = npy_dir / "qc"
        if not npy_dir.is_dir():
            continue
        qc_dir.mkdir(parents=True, exist_ok=True)
        assert npy_dir.resolve().is_relative_to(_root)
        for npy_path in sorted(npy_dir.glob("*.npy")):
            stem = npy_path.stem
            if _skip_noise_scale_stem(stem):
                continue
            noise_rng, scale_rng = _stem_rngs(seed, stem)
            arr = np.load(npy_path).astype(np.float64)
            lo, hi = float(np.nanmin(arr)), float(np.nanmax(arr))
            noisy = arr + noise_rng.normal(0.0, NOISE_SIGMA, size=arr.shape)
            noisy = np.nan_to_num(noisy, nan=0.0, posinf=hi, neginf=lo)
            noisy = noisy.astype(np.float32)
            stem_n = f"{stem}_aug_noise"
            np.save(npy_dir / f"{stem_n}.npy", noisy)
            save_qc_png(noisy, qc_dir / f"{stem_n}.png")
            n_noise += 1
            scale_factor = float(scale_rng.uniform(SCALE_LOW, SCALE_HIGH))
            scaled = np.clip(arr * scale_factor, lo, hi).astype(np.float32)
            sf_tag = f"{scale_factor:.3f}".replace(".", "p")
            stem_s = f"{stem}_aug_scale_{sf_tag}"
            np.save(npy_dir / f"{stem_s}.npy", scaled)
            save_qc_png(scaled, qc_dir / f"{stem_s}.png")
            n_scale += 1
    return n_noise, n_scale


print("Noise + scale on", AUG_OUT.resolve())
_nn_ns, _ns_ns = augment_pos_noise_and_scale(AUG_OUT, seed=AUGMENT_SEED)
print(
    f"Noise + scale: {_nn_ns} noise-augmented NPYs, {_ns_ns} scale-augmented NPYs "
    "(std + patch; each with qc/.png)"
)

Noise + scale on /home/zmirikha/Github/rnd_q2/data/ml_data_v4/pos
Noise + scale: 35200 noise-augmented NPYs, 35200 scale-augmented NPYs (std + patch; each with qc/.png)


In [8]:
# Two-panel chart: pos = dent/ counts per inspection; neg = stacked types per inspection (4 iids)
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

plt.style.use("dark_background")
try:
    _root = ML_DATA_V4
    _iids = INSPECTION_IDS
except NameError:
    _root = Path("/home/zmirikha/Github/rnd_q2/data/ml_data_v4")
    _iids = ["0A1OLRV96N3", "0A49KLT3B7Y", "0AGBXB4WEGN", "0ABP0TFUSH1"]


def _count_std_npy(d: Path) -> int:
    return len(list(d.glob("*.npy"))) if d.is_dir() else 0


def count_dent_per_iid(ml_root: Path, iid: str) -> int:
    return _count_std_npy(ml_root / "dent" / iid / "std_normalized")


def count_neg_type_iid(ml_root: Path, neg_type: str, iid: str) -> int:
    return _count_std_npy(ml_root / "neg" / neg_type / iid / "std_normalized")


# Up to 3 neg subfolders under neg/ (prefer girth_weld, random, fp; then any others)
_neg_root = _root / "neg"
_preferred = ["girth_weld", "random", "fp"]
_existing = {p.name for p in _neg_root.iterdir()} if _neg_root.is_dir() else set()
NEG_TYPES = [t for t in _preferred if t in _existing]
for t in sorted(_existing):
    if t not in NEG_TYPES and len(NEG_TYPES) < 3:
        NEG_TYPES.append(t)
NEG_TYPES = NEG_TYPES[:3]

fig, (ax_pos, ax_neg) = plt.subplots(1, 2, figsize=(14, 5))

# --- pos: dent ---
pos_h = [count_dent_per_iid(_root, iid) for iid in _iids]
xp = np.arange(len(_iids))
ax_pos.bar(xp, pos_h, color="#2ecc71", width=0.62)
ax_pos.set_xticks(xp)
ax_pos.set_xticklabels(_iids, rotation=22, ha="right")
ax_pos.set_ylabel("NPY count (std_normalized)")
ax_pos.set_title("pos — dent/ per inspection")
ax_pos.grid(axis="y", alpha=0.3)

# --- neg: one stacked bar per inspection (types stacked) ---
xn = np.arange(len(_iids))
w = 0.62
cmap = plt.cm.Set2(np.linspace(0, 1, max(len(NEG_TYPES), 1)))
bottom = np.zeros(len(_iids))
for j, ntype in enumerate(NEG_TYPES):
    h = np.array([
        count_neg_type_iid(_root, ntype, iid) / (2 if ntype == "girth_weld" else 1)
        for iid in _iids
    ], dtype=float)
    ax_neg.bar(xn, h, w, bottom=bottom, label=ntype + (" (÷2)" if ntype == "girth_weld" else ""), color=cmap[j % len(cmap)])
    bottom = bottom + h
ax_neg.set_xticks(xn)
ax_neg.set_xticklabels(_iids, rotation=22, ha="right")
ax_neg.set_ylabel("NPY count (std_normalized)")
ax_neg.set_title("neg — types stacked per inspection")
ax_neg.legend(title="neg type", loc="upper right", fontsize=8)
ax_neg.grid(axis="y", alpha=0.3)

fig.suptitle("ml_data_v4 — dent (pos) vs neg", y=1.03, fontsize=12)
plt.tight_layout()
_pos_dir = _root / "pos"
_pos_dir.mkdir(parents=True, exist_ok=True)
out_path = _pos_dir / "ml_data_v4_bar_chart.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved (under pos/, not dent/):", out_path)
print("  neg stacked types:", NEG_TYPES if NEG_TYPES else "(none under neg/)")

Saved (under pos/, not dent/): /home/zmirikha/Github/rnd_q2/data/ml_data_v4/pos/ml_data_v4_bar_chart.png
  neg stacked types: ['girth_weld', 'random', 'fp']


In [7]:
# Pipeline integrity check.
# After running cells 3 and 6 (rolls, then noise/scale), every NPY in
# std_normalized must have a 1:1 counterpart in patch_normalized and a
# matching QC PNG in each modality's qc/ folder.
def _stems(d: Path, suffix: str) -> set[str]:
    return {f.stem for f in d.glob(f"*{suffix}")} if d.is_dir() else set()


std_npy   = _stems(AUG_OUT / "std_normalized",   ".npy")
patch_npy = _stems(AUG_OUT / "patch_normalized", ".npy")
std_png   = _stems(AUG_OUT / "std_normalized" / "qc",   ".png")
patch_png = _stems(AUG_OUT / "patch_normalized" / "qc", ".png")

issues: list[str] = []
if std_npy != patch_npy:
    only_std   = sorted(std_npy   - patch_npy)
    only_patch = sorted(patch_npy - std_npy)
    issues.append(
        f"std vs patch NPY drift: {len(only_std)} only-in-std, "
        f"{len(only_patch)} only-in-patch"
    )
    for label, names in [("only_std", only_std), ("only_patch", only_patch)]:
        if names:
            print(f"    [{label} examples] " + ", ".join(names[:3]))
if std_npy != std_png:
    issues.append(f"std NPY/PNG drift: {len(std_npy ^ std_png)} stems differ")
if patch_npy != patch_png:
    issues.append(f"patch NPY/PNG drift: {len(patch_npy ^ patch_png)} stems differ")

if issues:
    print("INTEGRITY ISSUES:")
    for msg in issues:
        print(" -", msg)
else:
    print(
        f"Pipeline integrity OK: {len(std_npy)} stems × 4 sets equal "
        f"(std/patch × NPY/PNG)."
    )

Pipeline integrity OK: 52800 stems × 4 sets equal (std/patch × NPY/PNG).
